
### Data-Quality Elasticity: Benchmarking and Monitoring Digital-Twin Reliability Under Sensing Degradation 

**Runtime:** Google Colab · NVIDIA **L4** (`Runtime → Change runtime type → L4 GPU`). Fully **resumable** via Google Drive — run in as many sessions as needed; `Run all` after any disconnect continues where it stopped.

| § | Content |
|---|---|
| 1–3 | Environment/Drive · ingestion · multi-subset preprocessing |
| 4 | Injectors (+ mixed-degradation trainer) · tuned models · metrics/stats utilities |
| 5 | Resumable training · per-subset baselines with sanity gates |
| 6 | Multi-subset sweep · elasticity · breakpoints · cross-dataset consistency · H2 (Holm) · signature confirmation |
| 7 | Injection-point study · QCG ablation |
| 8 | Publication figures & tables · DQ monitor · submission checklist |

> **Compute:** full default (2 subsets × 3 arch × 5 sweep seeds, +8-seed interactions, +injection-point, +QCG) ≈ **4–8 GPU-hours total**, safely spread over sessions by the resume system. `QUICK_MODE=True` gives a ~1 h smoke pass.

In [ ]:
# ============================================================================
# SECTION 1 — ENVIRONMENT + GOOGLE DRIVE PERSISTENCE
# ============================================================================
import os
import subprocess
import sys


def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                   check=False)


pip_install(["seaborn>=0.13", "statsmodels>=0.14", "scikit-learn>=1.4",
             "tqdm>=4.66"])

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    RUN_ROOT = "/content/drive/MyDrive/DQ4DT"
    print("Google Drive mounted — artifacts persist across disconnects.")
except Exception as exc:  # noqa: BLE001
    RUN_ROOT = "/content/dq4dt_local"
    print(f"Drive unavailable ({exc}); using local (non-persistent) storage.")

CKPT_DIR = os.path.join(RUN_ROOT, "checkpoints")
FIGDIR = os.path.join(RUN_ROOT, "figures")
TBLDIR = os.path.join(RUN_ROOT, "tables")
DATA_ROOT = os.path.join(RUN_ROOT, "data")
for d in (RUN_ROOT, CKPT_DIR, FIGDIR, TBLDIR, DATA_ROOT):
    os.makedirs(d, exist_ok=True)

import torch

print("Torch  :", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), (
    "No GPU visible. Runtime > Change runtime type > L4 GPU.")
print("GPU    :", torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision("high")
print("Run root:", RUN_ROOT)


In [ ]:
# ============================================================================
# SECTION 1b — DETERMINISTIC SEEDING
# ============================================================================
import random

import numpy as np

GLOBAL_SEED = 42


def set_determinism(seed: int = GLOBAL_SEED, strict: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if strict:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        torch.use_deterministic_algorithms(False)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_determinism(GLOBAL_SEED, strict=True)
DEVICE = torch.device("cuda")
print("Seeded.")


## Section 2 — Ingestion (Drive-cached, auth-free cascade)

Verified sources: NASA Open Data Portal zip → PHM Society S3 mirror → GitHub raw mirror, with Kaggle fallback instructions on total failure. Cached once per Drive account. Integrity checks: all 12 files, 26-column parse, RUL length = #test engines.

In [ ]:
# ============================================================================
# SECTION 2 — C-MAPSS INGESTION
# ============================================================================
import io
import urllib.request
import zipfile

import pandas as pd

CMAPSS_DIR = os.path.join(DATA_ROOT, "CMAPSSData")
os.makedirs(CMAPSS_DIR, exist_ok=True)
ALL_SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
EXPECTED = [f"{k}_{s}.txt" for s in ALL_SUBSETS
            for k in ("train", "test", "RUL")]
UA = {"User-Agent": "Mozilla/5.0 (DQ4DT research notebook)"}


def _have_all_cmapss() -> bool:
    return all(os.path.exists(os.path.join(CMAPSS_DIR, f)) for f in EXPECTED)


def _extract_zip_bytes(raw: bytes, dest: str) -> None:
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for name in z.namelist():
            low = name.lower()
            if low.endswith(".zip"):
                _extract_zip_bytes(z.read(name), dest)
            elif low.endswith(".txt"):
                base = os.path.basename(name)
                if base:
                    with open(os.path.join(dest, base), "wb") as fh:
                        fh.write(z.read(name))


def _try_zip(url: str) -> bool:
    try:
        print(f"  trying zip: {url}")
        req = urllib.request.Request(url, headers=UA)
        raw = urllib.request.urlopen(req, timeout=180).read()
        _extract_zip_bytes(raw, CMAPSS_DIR)
        return _have_all_cmapss()
    except Exception as exc:  # noqa: BLE001
        print(f"    failed: {exc}")
        return False


def _try_github_rawfiles() -> bool:
    base = "https://raw.githubusercontent.com/edwardzjl/CMAPSSData/master/"
    ok = True
    for fname in EXPECTED:
        try:
            req = urllib.request.Request(base + fname, headers=UA)
            data = urllib.request.urlopen(req, timeout=60).read()
            with open(os.path.join(CMAPSS_DIR, fname), "wb") as fh:
                fh.write(data)
        except Exception as exc:  # noqa: BLE001
            print(f"    {fname} failed: {exc}")
            ok = False
    return ok and _have_all_cmapss()


def ingest_cmapss() -> None:
    if _have_all_cmapss():
        print("C-MAPSS: Drive cache hit — all 12 files present.")
        return
    for label, fetch in [
            ("NASA Open Data Portal",
             lambda: _try_zip(
                 "https://data.nasa.gov/docs/legacy/CMAPSSData.zip")),
            ("PHM Society S3 mirror",
             lambda: _try_zip(
                 "https://phm-datasets.s3.amazonaws.com/NASA/"
                 "6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip")),
            ("GitHub raw mirror", _try_github_rawfiles)]:
        print(f"[C-MAPSS] source: {label}")
        if fetch():
            print("  SUCCESS")
            return
    raise RuntimeError(
        "All auth-free sources failed. Kaggle fallback:\n"
        "  !pip -q install kaggle && mkdir -p ~/.kaggle && "
        "cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json\n"
        f"  !kaggle datasets download -d behrad3d/nasa-cmaps -p {CMAPSS_DIR}"
        " --unzip")


ingest_cmapss()

COLS = (["unit", "cycle"] + [f"op{i}" for i in (1, 2, 3)]
        + [f"s{i}" for i in range(1, 22)])


def load_cmapss_raw(subset: str):
    def _read(kind):
        path = os.path.join(CMAPSS_DIR, f"{kind}_{subset}.txt")
        df = pd.read_csv(path, sep=r"\s+", header=None).dropna(axis=1,
                                                               how="all")
        assert df.shape[1] == 26, f"{path}: {df.shape[1]} cols"
        df.columns = COLS
        return df

    train, test = _read("train"), _read("test")
    rul = pd.read_csv(os.path.join(CMAPSS_DIR, f"RUL_{subset}.txt"),
                      sep=r"\s+", header=None).iloc[:, 0].values
    assert len(rul) == test["unit"].nunique()
    return train, test, rul


print("Ingestion OK:",
      {s: load_cmapss_raw(s)[0].shape for s in ("FD001", "FD004")})


## Section 3 — Multi-subset Preprocessing

The validated pipeline, now building a `DATASETS` dict over the study subsets. FD004 exercises the **per-regime normalization** path (KMeans k=6 over op-settings, z-score within regime, train-fit only). Piecewise-linear RUL capped at 125; test RUL reconstructed from end-of-record values; engine-level 80/20 split; 30-cycle windows.

In [ ]:
# ============================================================================
# SECTION 3 — PREPROCESSING + DATASET REGISTRY
# ============================================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

CONST_SENSORS = [1, 5, 6, 10, 16, 18, 19]
SENSOR_COLS = [f"s{i}" for i in range(1, 22) if i not in CONST_SENSORS]
OP_COLS = ["op1", "op2", "op3"]
FEAT_COLS = OP_COLS + SENSOR_COLS
WINDOW, STRIDE, RUL_CAP = 30, 1, 125

# Study subsets. Add "FD002"/"FD003" here for the four-subset camera-ready.
STUDY_SUBSETS = ["FD001", "FD004"]


def add_train_rul(df, cap=RUL_CAP):
    df = df.copy()
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    df["RUL"] = (max_cycle - df["cycle"]).clip(upper=cap).astype(np.float32)
    return df


def add_test_rul(df, rul_last, cap=RUL_CAP):
    df = df.copy()
    unit_ids = np.sort(df["unit"].unique())
    rul_map = dict(zip(unit_ids, rul_last))
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    df["RUL"] = (df["unit"].map(rul_map) + (max_cycle - df["cycle"]))
    df["RUL"] = df["RUL"].clip(upper=cap).astype(np.float32)
    return df


def fit_condition_scalers(train, multi_condition):
    if multi_condition:
        km = KMeans(n_clusters=6, n_init=10,
                    random_state=GLOBAL_SEED).fit(train[OP_COLS])
        labels = km.predict(train[OP_COLS])
        scalers = {r: StandardScaler().fit(train.loc[labels == r,
                                                     SENSOR_COLS])
                   for r in range(6)}
        return km, scalers
    return None, {0: StandardScaler().fit(train[SENSOR_COLS])}


def apply_scalers(df, km, scalers):
    df = df.copy()
    if km is not None:
        labels = km.predict(df[OP_COLS])
        for r, sc in scalers.items():
            mask = labels == r
            if mask.any():
                df.loc[mask, SENSOR_COLS] = sc.transform(
                    df.loc[mask, SENSOR_COLS])
    else:
        df[SENSOR_COLS] = scalers[0].transform(df[SENSOR_COLS])
    for col in OP_COLS:
        lo, hi = df[col].min(), df[col].max()
        df[col] = (df[col] - lo) / (hi - lo + 1e-8)
    return df


def make_windows(df, is_train):
    xs, ys = [], []
    for _, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        feats = g[FEAT_COLS].values.astype(np.float32)
        rul = g["RUL"].values.astype(np.float32)
        n = len(g)
        if is_train:
            for start in range(0, n - WINDOW + 1, STRIDE):
                xs.append(feats[start:start + WINDOW])
                ys.append(rul[start + WINDOW - 1])
        else:
            if n >= WINDOW:
                xs.append(feats[-WINDOW:])
            else:
                pad = np.repeat(feats[:1], WINDOW - n, axis=0)
                xs.append(np.vstack([pad, feats]))
            ys.append(rul[-1])
    return np.asarray(xs, np.float32), np.asarray(ys, np.float32)


def build_subset(subset: str) -> dict:
    train_raw, test_raw, rul_test = load_cmapss_raw(subset)
    multi = subset in ("FD002", "FD004")
    train_raw = add_train_rul(train_raw)
    test_raw = add_test_rul(test_raw, rul_test)
    km, scalers = fit_condition_scalers(train_raw, multi)
    train_s = apply_scalers(train_raw, km, scalers)
    test_s = apply_scalers(test_raw, km, scalers)
    units = train_s["unit"].unique()
    rng = np.random.default_rng(GLOBAL_SEED)
    rng.shuffle(units)
    n_val = max(1, int(0.2 * len(units)))
    val_units = set(units[:n_val])
    tr_units = set(units[n_val:])
    x_tr, y_tr = make_windows(train_s[train_s["unit"].isin(tr_units)], True)
    x_va, y_va = make_windows(train_s[train_s["unit"].isin(val_units)], True)
    x_te, y_te = make_windows(test_s, False)
    return dict(Xtr=x_tr, ytr=y_tr, Xva=x_va, yva=y_va, Xte=x_te, yte=y_te,
                n_feat=len(FEAT_COLS), subset=subset)


DATASETS = {s: build_subset(s) for s in STUDY_SUBSETS}
for s, d in DATASETS.items():
    print(f"{s}: train {d['Xtr'].shape}, test {d['Xte'].shape}, "
          f"test-RUL {d['yte'].min():.0f}→{d['yte'].max():.0f}")


## Section 4 — Methodology (findings-informed, pre-registered)

**Confirmed v2 findings carried as confirmatory hypotheses on FD004** (discovery on FD001 → confirmation on unseen data — the strongest evidence structure available here):
- **C1 (bias ≫ noise):** $\eta^{rel}_{bias} > \eta^{rel}_{noise}$ in every architecture (v2: 2.1–3.0 vs 0.3–0.8).
- **C2 (MNAR ≫ MCAR):** ~5–7× at equal missing fraction.
- **C3 (invariant collapse point):** the MNAR breakpoint (v2: $1-q \approx 0.39$ for all three architectures on FD001, SSE gain 0.94–0.99) recurs on FD004 within a pre-set tolerance (±0.10).
- **C4 (elasticity signatures — replaces the failed H3):** v2 *refuted* uniform PINN robustness; the discovered signature — PINN most robust to **consistency**, transformer most robust to **drift** and most fragile to **bias/MNAR** — is tested confirmatorily on FD004 via paired per-seed Wilcoxon.
- **C5 (mechanism-based interactions):** same-information degradations (MNAR × timeliness) are **sub**-additive; different-mechanism degradations (bias × consistency) trend **super**-additive. 8 seeds, Holm-corrected family.
- **C6 (silent miscalibration):** coverage decays with severity (v2: 0.88 → 0.57) — reported as calibration elasticity.

**New in v3:** the **injection-point** axis (train vs. inference vs. both) for the two most damaging dimensions, and the **QCG ablation** (gate vs. matched no-gate control under identical mixed-degradation training).

In [ ]:
# ============================================================================
# SECTION 4a — INJECTION OPERATORS + MIXED-DEGRADATION TRAINER
# ============================================================================


def _sev_scale(s: int, lo: float, hi: float) -> float:
    return lo + (hi - lo) * (max(1, s) - 1) / 4.0


def _ffill(x: torch.Tensor) -> torch.Tensor:
    out = x.clone()
    for t in range(1, out.shape[1]):
        cur = out[:, t, :]
        prev = out[:, t - 1, :]
        nanmask = torch.isnan(cur)
        cur[nanmask] = prev[nanmask]
        out[:, t, :] = cur
    out[torch.isnan(out)] = 0.0
    return out


def inject_accuracy(x, s, bias=False, gen=None):
    std = _sev_scale(s, 0.05, 0.60)
    noise = torch.randn(x.shape, generator=gen, device=x.device) * std
    xd = x + noise + (std if bias else 0.0)
    u = torch.full((x.shape[0], x.shape[2]), 1.0 - min(std, 1.0),
                   device=x.device)
    return xd, u


def _missing_mask(x, s, mech, gen):
    p = _sev_scale(s, 0.05, 0.50)
    if mech == "MCAR":
        return torch.rand(x.shape, generator=gen, device=x.device) < p
    if mech == "MAR":
        driver = x[..., :1]
        prob = torch.sigmoid(3.0 * (driver - driver.mean())) * (2.0 * p)
        return torch.rand(x.shape, generator=gen, device=x.device) < prob
    prob = torch.sigmoid(3.0 * (x - x.mean())) * (2.0 * p)
    return torch.rand(x.shape, generator=gen, device=x.device) < prob


def make_missing(x, s, mech="MCAR", gen=None):
    """RAW NaN window (pre-imputation) + mask — consumed by the DQ monitor."""
    mask = _missing_mask(x, s, mech, gen)
    xd = x.clone()
    xd[mask] = float("nan")
    return xd, mask


def inject_completeness(x, s, mech="MCAR", gen=None):
    xd_raw, mask = make_missing(x, s, mech, gen)
    return _ffill(xd_raw), 1.0 - mask.float().mean(dim=1)


def inject_timeliness(x, s, gen=None):
    delay = int(round(_sev_scale(s, 1, 8)))
    xd = torch.roll(x, shifts=delay, dims=1)
    xd[:, :delay, :] = x[:, :1, :]
    factor = int(round(_sev_scale(s, 1, 5)))
    if factor > 1:
        idx = (torch.arange(x.shape[1], device=x.device) // factor) * factor
        xd = xd[:, idx.clamp(max=x.shape[1] - 1), :]
    u = torch.full((x.shape[0], x.shape[2]),
                   1.0 - min((delay + factor) / 13.0, 1.0), device=x.device)
    return xd, u


def inject_consistency(x, s, gen=None):
    off = _sev_scale(s, 0.1, 1.0)
    n_feat = x.shape[2]
    affected = torch.rand(n_feat, generator=gen, device=x.device) < 0.5
    scale = torch.where(affected,
                        torch.tensor(1.0 + off, device=x.device),
                        torch.tensor(1.0, device=x.device))
    shift = torch.where(affected,
                        torch.tensor(off, device=x.device),
                        torch.tensor(0.0, device=x.device))
    u = (1.0 - off * affected.float()).unsqueeze(0).expand(
        x.shape[0], -1).clone()
    return x * scale + shift, u


def inject_drift(x, s, gen=None):
    mag = _sev_scale(s, 0.1, 1.0)
    ramp = torch.linspace(0.0, mag, x.shape[1],
                          device=x.device).view(1, -1, 1)
    return x + ramp, torch.full((x.shape[0], x.shape[2]),
                                1.0 - min(mag, 1.0), device=x.device)


INJECTORS = {
    "accuracy":          lambda x, s, g: inject_accuracy(x, s, False, g),
    "accuracy_bias":     lambda x, s, g: inject_accuracy(x, s, True, g),
    "completeness_MCAR": lambda x, s, g: inject_completeness(x, s, "MCAR", g),
    "completeness_MAR":  lambda x, s, g: inject_completeness(x, s, "MAR", g),
    "completeness_MNAR": lambda x, s, g: inject_completeness(x, s, "MNAR", g),
    "timeliness":        lambda x, s, g: inject_timeliness(x, s, g),
    "consistency":       lambda x, s, g: inject_consistency(x, s, g),
    "drift":             lambda x, s, g: inject_drift(x, s, g),
}


def degrade(x, dim, severity, gen=None):
    if severity == 0 or dim is None:
        return x, torch.ones(x.shape[0], x.shape[2], device=x.device)
    return INJECTORS[dim](x, severity, gen)


# ---- Mixed-degradation trainer (QCG ablation & robust-training studies) ----
MIX_DIMS = ["accuracy_bias", "completeness_MNAR", "consistency", "drift"]


def mixed_degrade(xb, gen):
    """Sample one (dimension, severity) per batch; ~1/6 of batches stay
    clean so the model also sees pristine data."""
    pick = int(torch.randint(0, len(MIX_DIMS) + 1, (1,),
                             generator=gen, device=xb.device).item())
    if pick == len(MIX_DIMS):
        return degrade(xb, None, 0, gen)
    sev = int(torch.randint(1, 6, (1,), generator=gen,
                            device=xb.device).item())
    return degrade(xb, MIX_DIMS[pick], sev, gen)


_probe = torch.from_numpy(DATASETS["FD001"]["Xte"][:8]).to(DEVICE)
_gen = torch.Generator(device=DEVICE)
_gen.manual_seed(0)
for _dim in INJECTORS:
    _xd, _u = degrade(_probe, _dim, 3, _gen)
    assert _xd.shape == _probe.shape and not torch.isnan(_xd).any()
_xm, _um = mixed_degrade(_probe, _gen)
assert _xm.shape == _probe.shape
print("Injector + mixed-trainer self-test passed.")


### 4b. Architectures (v3: transformer tuned)

| Model | v3 change | Rationale |
|---|---|---|
| `LSTMTwin` | — | healthy in v2 (16.0) |
| `TransformerTwin` | **d_model 96→128, mean+last-token pooling, lr 5e-4** | v2's weakest clean baseline (18.4 vs 15) — short 30-step windows under-use mean-only pooling; lower lr stabilizes the fast early convergence seen in the v2 audit |
| `PINNTwin` | — | v2's best baseline (13.5) after the GroupNorm fix |

Per-model learning rates live in `LR` (Section 5). Everything else — SDPA flash attention, gradient checkpointing, BF16, heteroscedastic heads, optional QCG — unchanged.

In [ ]:
# ============================================================================
# SECTION 4b — MODELS
# ============================================================================
import torch.nn as nn
import torch.nn.functional as F_
import torch.utils.checkpoint as ckpt


class QCG(nn.Module):
    def __init__(self, n_feat: int, hidden: int = 32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_feat, hidden), nn.GELU(),
                                 nn.Linear(hidden, n_feat), nn.Sigmoid())

    def forward(self, x, u):
        return x * self.net(u).unsqueeze(1)


class HeteroHead(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.mu = nn.Linear(d, 1)
        self.lv = nn.Linear(d, 1)

    def forward(self, h):
        return self.mu(h).squeeze(-1), self.lv(h).squeeze(-1).clamp(-6.0, 6.0)


class LSTMTwin(nn.Module):
    def __init__(self, n_feat, hidden=96, layers=2, use_qcg=False):
        super().__init__()
        self.qcg = QCG(n_feat) if use_qcg else None
        self.rnn = nn.LSTM(n_feat, hidden, layers, batch_first=True,
                           dropout=0.2)
        self.head = HeteroHead(hidden)

    def forward(self, x, u=None):
        if self.qcg is not None and u is not None:
            x = self.qcg(x, u)
        out, _ = self.rnn(x)
        return self.head(out[:, -1])


class TransformerTwin(nn.Module):
    """v3: d_model 128, mean+last pooling. SDPA flash backend under BF16."""

    def __init__(self, n_feat, d_model=128, nhead=4, layers=2, use_qcg=False,
                 max_len=64):
        super().__init__()
        self.qcg = QCG(n_feat) if use_qcg else None
        self.proj = nn.Linear(n_feat, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=4 * d_model, dropout=0.1,
            batch_first=True, activation="gelu", norm_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, layers,
                                         enable_nested_tensor=False)
        self.head = HeteroHead(2 * d_model)      # mean (+) last-token pooling
        self.grad_ckpt = True

    def forward(self, x, u=None):
        if self.qcg is not None and u is not None:
            x = self.qcg(x, u)
        h = self.proj(x) + self.pos[:, :x.shape[1]]
        if self.grad_ckpt and self.training:
            h = ckpt.checkpoint(self.enc, h, use_reentrant=False)
        else:
            h = self.enc(h)
        pooled = torch.cat([h.mean(dim=1), h[:, -1]], dim=-1)
        return self.head(pooled)


class PINNTwin(nn.Module):
    def __init__(self, n_feat, ch=64, use_qcg=False):
        super().__init__()
        self.qcg = QCG(n_feat) if use_qcg else None
        self.tcn = nn.Sequential(
            nn.Conv1d(n_feat, ch, 5, padding=2), nn.GELU(),
            nn.GroupNorm(8, ch),
            nn.Conv1d(ch, ch, 5, padding=4, dilation=2), nn.GELU(),
            nn.GroupNorm(8, ch),
            nn.Conv1d(ch, ch, 5, padding=8, dilation=4), nn.GELU(),
            nn.GroupNorm(8, ch))
        self.head = HeteroHead(ch)

    def forward(self, x, u=None):
        if self.qcg is not None and u is not None:
            x = self.qcg(x, u)
        h = self.tcn(x.transpose(1, 2))
        return self.head(h.mean(dim=-1))


MODEL_REGISTRY = {"lstm": LSTMTwin, "transformer": TransformerTwin,
                  "pinn": PINNTwin}


def build_model(name, n_feat, use_qcg=False):
    return MODEL_REGISTRY[name](n_feat, use_qcg=use_qcg).to(DEVICE)


for _name in MODEL_REGISTRY:
    _m = build_model(_name, DATASETS["FD001"]["n_feat"])
    print(f"{_name:12s} "
          f"{sum(p.numel() for p in _m.parameters())/1e3:8.1f}k params")
    del _m


In [ ]:
# ============================================================================
# SECTION 4c — LOSSES, METRICS, CALIBRATION, STATISTICS UTILITIES
# ============================================================================
from scipy.stats import norm as _norm
from scipy.stats import wilcoxon


def gaussian_nll(mu, logvar, y):
    return 0.5 * (logvar + (y - mu) ** 2 / torch.exp(logvar)).mean()


def physics_penalty(mu_sorted):
    diff = mu_sorted[1:] - mu_sorted[:-1]
    return F_.relu(diff).mean() + F_.relu(-mu_sorted).mean()


def rmse(pred, true):
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def nasa_score(pred, true):
    d = np.clip(pred - true, -500.0, 500.0)
    s = np.where(d < 0, np.exp(np.clip(-d / 13.0, None, 50)) - 1.0,
                 np.exp(np.clip(d / 10.0, None, 50)) - 1.0)
    return float(np.sum(s))


def ece_regression(mu, sigma, y, n_bins: int = 10):
    ps = np.linspace(0.05, 0.95, n_bins)
    return float(np.mean([abs(np.mean(y <= mu + sigma * _norm.ppf(p)) - p)
                          for p in ps]))


def coverage(mu, sigma, y, level: float = 0.9):
    z = _norm.ppf(0.5 + level / 2.0)
    return float(np.mean((y >= mu - z * sigma) & (y <= mu + z * sigma)))


def fit_sigma_scale(mu, sigma, y) -> float:
    z2 = ((y - mu) / np.maximum(sigma, 1e-6)) ** 2
    return float(np.sqrt(np.mean(z2)))


# ---- Statistics utilities ---------------------------------------------------
def holm(pvals):
    """Holm step-down adjusted p-values (family-wise error control)."""
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


def rank_biserial(diffs):
    """Matched-pairs rank-biserial correlation (effect size for Wilcoxon)."""
    d = np.asarray(diffs, float)
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    ranks = np.argsort(np.argsort(np.abs(d))) + 1.0
    w_pos = ranks[d > 0].sum()
    w_neg = ranks[d < 0].sum()
    return float((w_pos - w_neg) / (w_pos + w_neg))


def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))


## Section 5 — Resumable Training & Per-subset Baselines

Unchanged resume backbone: per-epoch `last_*.pt` autosaves; instant-loading `final_*.pt` on completion; σ-calibration fitted on validation and stored in the checkpoint. Per-model learning rates (`LR`) implement the transformer fix. Checkpoints are keyed by **(subset, model, seed, qcg, tag)**, so FD001 and FD004 runs, injection-point models, and QCG-ablation models never collide.

In [ ]:
# ============================================================================
# SECTION 5 — TRAINING (per-epoch autosave -> Drive; full resume)
# ============================================================================
import copy
import math
import time

from torch.utils.data import DataLoader, TensorDataset

WARMUP_EPOCHS = 5
PHYSICS_W = 0.01
IMPROVE_MARGIN = 0.05
LR = {"lstm": 1e-3, "transformer": 5e-4, "pinn": 1e-3}   # v3 transformer fix


def make_loader(x, y, bs=256, shuffle=True):
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    gen = torch.Generator()
    gen.manual_seed(GLOBAL_SEED)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2,
                      pin_memory=True, persistent_workers=True,
                      prefetch_factor=4, worker_init_fn=seed_worker,
                      generator=gen, drop_last=shuffle)


def cosine_warmup(step, total, warmup, base_lr, min_lr=1e-5):
    if step < warmup:
        return base_lr * step / max(1, warmup)
    prog = (step - warmup) / max(1, total - warmup)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * prog))


@torch.no_grad()
def evaluate(model, loader, degrade_fn=None, gen=None, sigma_scale=1.0):
    model.eval()
    mus, sigs, ys = [], [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        u = None
        if degrade_fn is not None:
            xb, u = degrade_fn(xb, gen)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            mu, lv = model(xb, u)
        mus.append(mu.float().cpu().numpy())
        sigs.append(torch.exp(0.5 * lv).float().cpu().numpy())
        ys.append(yb.numpy())
    mu = np.concatenate(mus)
    sig = np.concatenate(sigs) * sigma_scale
    y = np.concatenate(ys)
    return dict(rmse=rmse(mu, y), score=nasa_score(mu, y),
                ece=ece_regression(mu, sig, y),
                cov90=coverage(mu, sig, y, 0.9), mu=mu, sigma=sig, y=y)


def _ckpt_paths(subset, name, seed, use_qcg, tag=""):
    stem = f"{subset}_{name}_s{seed}_qcg{int(use_qcg)}{tag}"
    return (os.path.join(CKPT_DIR, f"final_{stem}.pt"),
            os.path.join(CKPT_DIR, f"last_{stem}.pt"))


def train_model(name, data, use_qcg=False, epochs=80, patience=15,
                base_lr=None, use_physics=None, seed=GLOBAL_SEED,
                train_degrade_fn=None, tag="", verbose=True):
    final_p, last_p = _ckpt_paths(data["subset"], name, seed, use_qcg, tag)
    base_lr = LR[name] if base_lr is None else base_lr
    use_physics = (name == "pinn") if use_physics is None else use_physics

    if os.path.exists(final_p):
        blob = torch.load(final_p, map_location=DEVICE, weights_only=False)
        model = build_model(name, data["n_feat"], use_qcg)
        model.load_state_dict(blob["state"])
        if verbose:
            print(f"[{data['subset']} {name} s{seed}{tag}] final checkpoint "
                  f"loaded (best val {blob['best_rmse']:.2f}).")
        return model, blob

    set_determinism(seed, strict=False)
    model = build_model(name, data["n_feat"], use_qcg)
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=1e-4)
    tr = make_loader(data["Xtr"], data["ytr"], shuffle=True)
    va = make_loader(data["Xva"], data["yva"], shuffle=False)
    total_steps = epochs * len(tr)
    lr_warm = int(0.05 * total_steps)
    dgen = torch.Generator(device=DEVICE)
    dgen.manual_seed(seed)
    start_ep, step, bad = 0, 0, 0
    best_rmse, best_state, history = float("inf"), None, []

    if os.path.exists(last_p):
        blob = torch.load(last_p, map_location=DEVICE, weights_only=False)
        model.load_state_dict(blob["model"])
        opt.load_state_dict(blob["opt"])
        start_ep = blob["epoch"] + 1
        step = blob["step"]
        bad = blob["bad"]
        best_rmse = blob["best_rmse"]
        best_state = blob["best_state"]
        history = blob["history"]
        print(f"[{data['subset']} {name} s{seed}{tag}] RESUMING at epoch "
              f"{start_ep} (best {best_rmse:.2f}).")

    t0 = time.time()
    for ep in range(start_ep, epochs):
        model.train()
        warm = ep < WARMUP_EPOCHS
        for xb, yb in tr:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            u = None
            if train_degrade_fn is not None:
                xb, u = train_degrade_fn(xb, dgen)
            for pg in opt.param_groups:
                pg["lr"] = cosine_warmup(step, total_steps, lr_warm, base_lr)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                mu, lv = model(xb, u)
                if warm:
                    loss = F_.mse_loss(mu, yb)
                else:
                    loss = gaussian_nll(mu, lv, yb)
                    if use_physics:
                        order = torch.argsort(yb, descending=True)
                        loss = loss + PHYSICS_W * physics_penalty(mu[order])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step += 1
        val_rmse = evaluate(model, va)["rmse"]
        history.append(val_rmse)
        if val_rmse < best_rmse - IMPROVE_MARGIN:
            best_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
        if verbose and (ep % 10 == 0 or bad >= patience):
            print(f"[{data['subset']} {name} s{seed}{tag}] ep{ep:02d} "
                  f"val={val_rmse:6.2f} best={best_rmse:6.2f}")
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                        epoch=ep, step=step, bad=bad, best_rmse=best_rmse,
                        best_state=best_state, history=history), last_p)
        if bad >= patience and not warm:
            break

    model.load_state_dict(best_state)
    val_res = evaluate(model, va)
    s_scale = fit_sigma_scale(val_res["mu"], val_res["sigma"], val_res["y"])
    blob = dict(state=best_state, history=history, best_rmse=best_rmse,
                sigma_scale=s_scale)
    torch.save(blob, final_p)
    if os.path.exists(last_p):
        os.remove(last_p)
    if verbose:
        print(f"[{data['subset']} {name} s{seed}{tag}] done "
              f"({time.time()-t0:.0f}s) best {best_rmse:.2f} "
              f"sigma-scale {s_scale:.2f}")
    return model, blob


_MODEL_CACHE = {}


def get_trained(name, data, seed=GLOBAL_SEED, use_qcg=False, **kw):
    key = (data["subset"], name, seed, use_qcg, kw.get("tag", ""))
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = train_model(name, data, use_qcg=use_qcg,
                                        seed=seed, **kw)
    return _MODEL_CACHE[key]


In [ ]:
# ============================================================================
# SECTION 5b — CONFIG + PER-SUBSET BASELINES (sanity gates)
# ============================================================================
QUICK_MODE = False

CONFIG = dict(
    subsets=STUDY_SUBSETS if not QUICK_MODE else ["FD001"],
    dims=["accuracy", "accuracy_bias", "completeness_MCAR",
          "completeness_MAR", "completeness_MNAR", "timeliness",
          "consistency", "drift"],
    severities=[1, 3, 5] if QUICK_MODE else [1, 2, 3, 4, 5],
    models=["lstm", "transformer", "pinn"],
    seeds=[0, 1, 2] if QUICK_MODE else [0, 1, 2, 3, 4],
    interaction_seeds=[0, 1, 2] if QUICK_MODE else list(range(8)),
)
print(f"CONFIG: subsets={CONFIG['subsets']} | "
      f"{len(CONFIG['models'])} arch x {len(CONFIG['seeds'])} seeds x "
      f"{len(CONFIG['dims'])} dims x {len(CONFIG['severities'])} sevs | "
      f"interaction seeds={len(CONFIG['interaction_seeds'])} "
      f"(QUICK_MODE={QUICK_MODE})")

TEST_LOADERS = {s: make_loader(d["Xte"], d["yte"], shuffle=False)
                for s, d in DATASETS.items()}

CLEAN = {s: {} for s in CONFIG["subsets"]}
for _sub in CONFIG["subsets"]:
    for _name in CONFIG["models"]:
        _mdl, _meta = get_trained(_name, DATASETS[_sub], seed=GLOBAL_SEED)
        _pre = evaluate(_mdl, TEST_LOADERS[_sub])
        _post = evaluate(_mdl, TEST_LOADERS[_sub],
                         sigma_scale=_meta["sigma_scale"])
        CLEAN[_sub][_name] = dict(
            model=_mdl, meta=_meta, rmse=_post["rmse"], score=_post["score"],
            ece_pre=_pre["ece"], ece_post=_post["ece"],
            cov_pre=_pre["cov90"], cov_post=_post["cov90"])
        print(f"{_sub} {_name:12s} RMSE={_post['rmse']:6.2f} "
              f"NASA={_post['score']:10.0f} "
              f"ECE {_pre['ece']:.3f}->{_post['ece']:.3f} "
              f"cov@90 {_pre['cov90']:.2f}->{_post['cov90']:.2f}")

# Sanity gates (FD004 is genuinely harder — wider bound).
for _sub in CONFIG["subsets"]:
    bound = 30 if _sub in ("FD001", "FD003") else 45
    for _name in CONFIG["models"]:
        assert CLEAN[_sub][_name]["rmse"] < bound, (
            f"{_sub}/{_name} failed its sanity gate — inspect "
            f"CLEAN['{_sub}']['{_name}']['meta']['history']")
print("\nAll baselines passed their sanity gates.")


## Section 6 — Core Studies

The sweep now spans **subsets × architectures × seeds × dimensions × severities**, resumable at `(subset, model, seed)` granularity. Downstream: per-subset elasticity with bootstrap CIs → breakpoints → **cross-dataset consistency table (C3)** → **H2/C5 interactions** (8 seeds, Holm, rank-biserial) → **C4 signature confirmation on FD004**.

In [ ]:
# ============================================================================
# SECTION 6a — MULTI-SUBSET RESUMABLE SWEEP
# ============================================================================
from tqdm.auto import tqdm

SWEEP_CSV = os.path.join(RUN_ROOT, "sweep_results_v3.csv")


def mean_quality_for(dim, severity, probe_x):
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(0)
    _, u = degrade(probe_x, dim, severity, gen)
    return float(u.mean().cpu())


def run_sweep(cfg=CONFIG):
    if os.path.exists(SWEEP_CSV):
        done_df = pd.read_csv(SWEEP_CSV)
        expected = len(cfg["dims"]) * (len(cfg["severities"]) + 1)
        done_keys = {k for k, g in
                     done_df.groupby(["subset", "model", "seed"])
                     if len(g) >= expected}
        print(f"Sweep resume: {len(done_keys)} completed blocks on Drive.")
    else:
        done_df = pd.DataFrame()
        done_keys = set()

    combos = [(sub, m, s) for sub in cfg["subsets"]
              for m in cfg["models"] for s in cfg["seeds"]]
    for sub, model_name, seed in tqdm(combos, desc="subset x arch x seed"):
        if (sub, model_name, seed) in done_keys:
            continue
        data = DATASETS[sub]
        te = TEST_LOADERS[sub]
        probe = torch.from_numpy(data["Xte"][:256]).to(DEVICE)
        mdl, meta = get_trained(model_name, data, seed=seed, verbose=False)
        ss = meta["sigma_scale"]
        base = evaluate(mdl, te, sigma_scale=ss)
        block = []
        for dim in cfg["dims"]:
            for sev in [0] + cfg["severities"]:
                gen = torch.Generator(device=DEVICE)
                gen.manual_seed(1000 + 31 * sev + sum(map(ord, dim)) % 997)
                deg_fn = (lambda xb, g, d=dim, s=sev: degrade(xb, d, s, g))
                res = evaluate(mdl, te, degrade_fn=deg_fn, gen=gen,
                               sigma_scale=ss)
                q = 1.0 if sev == 0 else mean_quality_for(dim, sev, probe)
                block.append(dict(subset=sub, model=model_name, seed=seed,
                                  dim=dim, sev=sev, one_minus_q=1.0 - q,
                                  rmse=res["rmse"], score=res["score"],
                                  ece=res["ece"], cov90=res["cov90"],
                                  rmse_base=base["rmse"]))
        done_df = pd.concat([done_df, pd.DataFrame(block)],
                            ignore_index=True)
        done_df.to_csv(SWEEP_CSV, index=False)
    return done_df


SWEEP = run_sweep()
print(f"Sweep rows: {len(SWEEP)} -> {SWEEP_CSV}")


In [ ]:
# ============================================================================
# SECTION 6b — ELASTICITY + BREAKPOINTS + CROSS-DATASET CONSISTENCY (C3)
# ============================================================================

def compute_elasticity(df, fidelity="rmse", n_boot=500):
    rows = []
    rng = np.random.default_rng(GLOBAL_SEED)
    for (sub, m, d), g in df.groupby(["subset", "model", "dim"]):
        agg = g.groupby("one_minus_q")[fidelity].mean().reset_index()
        if len(agg) < 2:
            continue
        slope = np.polyfit(agg["one_minus_q"], agg[fidelity], 1)[0]
        base = g["rmse_base"].mean()
        seeds = g["seed"].unique()
        boots = []
        for _ in range(n_boot):
            samp = rng.choice(seeds, len(seeds), replace=True)
            s2 = (pd.concat([g[g.seed == s] for s in samp])
                  .groupby("one_minus_q")[fidelity].mean().reset_index())
            if len(s2) >= 2:
                boots.append(np.polyfit(s2["one_minus_q"],
                                        s2[fidelity], 1)[0])
        lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots
                  else (slope, slope))
        rows.append(dict(subset=sub, model=m, dim=d, elasticity=slope,
                         ci_lo=lo, ci_hi=hi, elasticity_rel=slope / base,
                         rel_ci_lo=lo / base, rel_ci_hi=hi / base))
    return pd.DataFrame(rows).sort_values(["subset", "dim", "model"])


ELAST = compute_elasticity(SWEEP, "rmse")
ELAST.to_csv(os.path.join(RUN_ROOT, "elasticity_v3.csv"), index=False)
for sub in CONFIG["subsets"]:
    print(f"\n=== {sub} normalized elasticity ===")
    print(ELAST[ELAST.subset == sub]
          .pivot(index="model", columns="dim", values="elasticity_rel")
          .round(2).to_string())


def fit_breakpoint(x, y):
    x, y = np.asarray(x), np.asarray(y)
    lin_res = np.polyfit(x, y, 1, full=True)
    sse_lin = float(lin_res[1][0]) if len(lin_res[1]) else 0.0
    best = (None, np.inf)
    for brk in x[1:-1]:
        design = np.column_stack(
            [np.ones_like(x), x, np.maximum(x - brk, 0.0)])
        _, res, _, _ = np.linalg.lstsq(design, y, rcond=None)
        sse = float(res[0]) if len(res) else 0.0
        if sse < best[1]:
            best = (float(brk), sse)
    return best[0], best[1], sse_lin


def breakpoint_table(df, min_gain=0.2):
    rows = []
    for (sub, m, d), g in df.groupby(["subset", "model", "dim"]):
        agg = g.groupby("one_minus_q")["rmse"].mean().reset_index()
        if len(agg) < 4:
            continue
        brk, sse_seg, sse_lin = fit_breakpoint(agg["one_minus_q"],
                                               agg["rmse"])
        gain = 1.0 - sse_seg / (sse_lin + 1e-12)
        rows.append(dict(subset=sub, model=m, dim=d,
                         breakpoint=(round(brk, 3)
                                     if (brk is not None and gain >= min_gain)
                                     else np.nan),
                         sse_gain=round(gain, 2)))
    return pd.DataFrame(rows).sort_values(["dim", "subset", "model"])


BREAKS = breakpoint_table(SWEEP)
BREAKS.to_csv(os.path.join(RUN_ROOT, "breakpoints_v3.csv"), index=False)

# ---- C3: cross-dataset breakpoint consistency (tolerance +/- 0.10) ---------
C3_TOL = 0.10
c3_rows = []
for dim in ("completeness_MNAR", "accuracy_bias"):
    sub_means = (BREAKS[BREAKS.dim == dim].dropna(subset=["breakpoint"])
                 .groupby("subset")["breakpoint"].mean())
    if len(sub_means) >= 2:
        spread = float(sub_means.max() - sub_means.min())
        c3_rows.append(dict(dim=dim,
                            **{f"brk_{s}": round(v, 3)
                               for s, v in sub_means.items()},
                            spread=round(spread, 3),
                            consistent=bool(spread <= C3_TOL)))
C3 = pd.DataFrame(c3_rows)
C3.to_csv(os.path.join(RUN_ROOT, "c3_cross_dataset.csv"), index=False)
print("\nC3 — cross-dataset breakpoint consistency (tolerance "
      f"±{C3_TOL}):")
print(C3.to_string(index=False) if len(C3) else
      "  (needs >= 2 subsets in the sweep)")


In [ ]:
# ============================================================================
# SECTION 6c — C5/H2 INTERACTIONS (8 seeds, Holm, rank-biserial)
#              + C4 SIGNATURE CONFIRMATION ON FD004
# ============================================================================

def interaction_test(model_name, dim_a, dim_b, sev=5, subset="FD001",
                     seeds=None):
    seeds = CONFIG["interaction_seeds"] if seeds is None else seeds
    data = DATASETS[subset]
    te = TEST_LOADERS[subset]

    def chained(dims):
        def fn(xb, g):
            u = torch.ones(xb.shape[0], xb.shape[2], device=xb.device)
            for d in dims:
                xb2, ui = degrade(xb, d, sev, g)
                xb = xb2
                u = torch.minimum(u, ui)
            return xb, u
        return fn

    per_seed = []
    for seed in seeds:
        mdl, meta = get_trained(model_name, data, seed=seed, verbose=False)
        ss = meta["sigma_scale"]
        base = evaluate(mdl, te, sigma_scale=ss)["rmse"]
        gen = torch.Generator(device=DEVICE)
        gen.manual_seed(7 + seed)
        d_a = evaluate(mdl, te, chained([dim_a]), gen, ss)["rmse"] - base
        d_b = evaluate(mdl, te, chained([dim_b]), gen, ss)["rmse"] - base
        d_ab = evaluate(mdl, te, chained([dim_a, dim_b]), gen,
                        ss)["rmse"] - base
        per_seed.append(dict(seed=seed, dA=d_a, dB=d_b, dAB=d_ab,
                             gap=d_ab - (d_a + d_b)))
    df = pd.DataFrame(per_seed)
    gaps = df["gap"].values
    direction = "greater" if np.mean(gaps) > 0 else "less"
    p = (wilcoxon(gaps, alternative=direction).pvalue
         if len(gaps) >= 3 and np.any(gaps != 0) else np.nan)
    return dict(subset=subset, model=model_name, dim_a=dim_a, dim_b=dim_b,
                mean_gap=float(np.mean(gaps)), p_raw=float(p),
                r_rb=rank_biserial(gaps),
                verdict=("SUPER-additive" if np.mean(gaps) > 0
                         else "SUB-additive"),
                per_seed=df)


INTERACTIONS = [
    interaction_test("lstm", "completeness_MNAR", "timeliness"),
    interaction_test("transformer", "completeness_MNAR", "timeliness"),
    interaction_test("lstm", "accuracy_bias", "consistency"),
    interaction_test("transformer", "accuracy_bias", "consistency"),
]
p_adj = holm([r["p_raw"] for r in INTERACTIONS])
for r, pa in zip(INTERACTIONS, p_adj):
    r["p_holm"] = float(pa)
    print(f"C5 [{r['dim_a']} x {r['dim_b']} | {r['model']}] "
          f"gap {r['mean_gap']:+.2f} -> {r['verdict']} "
          f"(p_raw={r['p_raw']:.4f}, p_Holm={r['p_holm']:.4f}, "
          f"r_rb={r['r_rb']:+.2f}, n={len(r['per_seed'])})")
INTER_DF = pd.DataFrame([{k: v for k, v in r.items() if k != "per_seed"}
                         for r in INTERACTIONS])
INTER_DF.to_csv(os.path.join(RUN_ROOT, "interactions_v3.csv"), index=False)


# ---- C4: signature confirmation (discovery=FD001 -> confirmation=FD004) ----
def per_seed_rel_slopes(subset, model, dim):
    out = []
    for s in CONFIG["seeds"]:
        g = SWEEP[(SWEEP.subset == subset) & (SWEEP.model == model)
                  & (SWEEP.dim == dim) & (SWEEP.seed == s)]
        agg = g.groupby("one_minus_q")["rmse"].mean().reset_index()
        if len(agg) >= 2:
            slope = np.polyfit(agg["one_minus_q"], agg["rmse"], 1)[0]
            out.append(slope / g["rmse_base"].mean())
    return np.array(out)


SIGNATURE_TESTS = [
    # (dim, arch expected MORE robust, arch expected LESS robust)
    ("consistency", "pinn", "lstm"),
    ("consistency", "pinn", "transformer"),
    ("drift", "transformer", "lstm"),
    ("drift", "transformer", "pinn"),
    ("accuracy_bias", "lstm", "transformer"),
    ("completeness_MNAR", "lstm", "transformer"),
]


def c4_confirmation(subset):
    rows = []
    for dim, robust, fragile in SIGNATURE_TESTS:
        a = per_seed_rel_slopes(subset, robust, dim)
        b = per_seed_rel_slopes(subset, fragile, dim)
        n = min(len(a), len(b))
        diff = a[:n] - b[:n]
        p = (wilcoxon(diff, alternative="less").pvalue
             if n >= 3 and np.any(diff != 0) else np.nan)
        rows.append(dict(subset=subset, dim=dim, robust=robust,
                         fragile=fragile, eta_robust=round(a.mean(), 2),
                         eta_fragile=round(b.mean(), 2),
                         confirmed=bool(a.mean() < b.mean()),
                         p_raw=float(p)))
    df = pd.DataFrame(rows)
    df["p_holm"] = holm(df["p_raw"].values)
    return df


C4_FRAMES = [c4_confirmation(s) for s in CONFIG["subsets"]]
C4 = pd.concat(C4_FRAMES, ignore_index=True)
C4.to_csv(os.path.join(RUN_ROOT, "c4_signatures.csv"), index=False)
print("\nC4 — architecture elasticity-signature tests "
      "(FD001 = discovery, FD004 = confirmation):")
print(C4.round({"p_raw": 4, "p_holm": 4}).to_string(index=False))


## Section 7 — Injection-Point Study & QCG Ablation

**Injection points (completes the lifecycle claim).** For the two most damaging dimensions (`accuracy_bias`, `completeness_MNAR`): *train-point* trains on degraded data and tests clean (does bad historical data poison the twin?); *both-point* trains and tests degraded (the realistic deployed case); *inference-point* values come from the main sweep. One training per (dim, sev, seed) covers both new rows; everything resumes via tags.

**QCG ablation (fair by construction).** Gate-on vs. gate-off models are trained under *identical* mixed degradation (random dimension × severity per batch, ~1/6 clean), same seeds, same budget — isolating the gate itself. Outcome: per-dimension elasticity difference, paired Wilcoxon over seeds, Holm-corrected. If the gate does not help, that null is reported and QCG is framed as a negative result — not silently dropped.

In [ ]:
# ============================================================================
# SECTION 7a — INJECTION-POINT STUDY (resumable via checkpoint tags)
# ============================================================================
IP_CONFIG = dict(subset="FD001", model="lstm",
                 dims=["accuracy_bias", "completeness_MNAR"],
                 sevs=[3, 5], seeds=[0, 1, 2])
IP_CSV = os.path.join(RUN_ROOT, "injection_points_v3.csv")


def run_injection_point_study(cfg=IP_CONFIG):
    data = DATASETS[cfg["subset"]]
    te = TEST_LOADERS[cfg["subset"]]
    done = (pd.read_csv(IP_CSV) if os.path.exists(IP_CSV)
            else pd.DataFrame())
    done_keys = ({(r.dim, r.sev, r.seed) for r in done.itertuples()}
                 if len(done) else set())
    rows = []
    for dim in cfg["dims"]:
        for sev in cfg["sevs"]:
            for seed in cfg["seeds"]:
                if (dim, sev, seed) in done_keys:
                    continue
                # Plain model -> clean base + inference-point row.
                base_mdl, base_meta = get_trained(cfg["model"], data,
                                                  seed=seed, verbose=False)
                bss = base_meta["sigma_scale"]
                clean = evaluate(base_mdl, te, sigma_scale=bss)["rmse"]
                gen = torch.Generator(device=DEVICE)
                gen.manual_seed(77 + seed)
                deg_fn = (lambda xb, g, d=dim, s=sev: degrade(xb, d, s, g))
                inf_r = evaluate(base_mdl, te, deg_fn, gen, bss)["rmse"]
                # Degraded-train model -> train-point + both-point rows.
                tag = f"_tp_{dim}{sev}"
                tp_fn = (lambda xb, g, d=dim, s=sev: degrade(xb, d, s, g))
                mdl, meta = get_trained(cfg["model"], data, seed=seed,
                                        tag=tag, train_degrade_fn=tp_fn,
                                        verbose=False)
                ss = meta["sigma_scale"]
                train_r = evaluate(mdl, te, sigma_scale=ss)["rmse"]
                gen2 = torch.Generator(device=DEVICE)
                gen2.manual_seed(77 + seed)
                both_r = evaluate(mdl, te, deg_fn, gen2, ss)["rmse"]
                rows.append(dict(dim=dim, sev=sev, seed=seed,
                                 clean=clean,
                                 d_inference=inf_r - clean,
                                 d_train=train_r - clean,
                                 d_both=both_r - clean))
                done = pd.concat([done, pd.DataFrame(rows[-1:])],
                                 ignore_index=True)
                done.to_csv(IP_CSV, index=False)
    return done


IP = run_injection_point_study()
print("Injection-point study (dRMSE vs clean, mean over seeds):")
print(IP.groupby(["dim", "sev"])[["d_inference", "d_train", "d_both"]]
      .mean().round(2).to_string())


In [ ]:
# ============================================================================
# SECTION 7b — QCG ABLATION (gate vs matched no-gate control)
# ============================================================================
QCG_CONFIG = dict(subset="FD001", model="lstm", seeds=[0, 1, 2],
                  dims=MIX_DIMS, sevs=[1, 3, 5])


def qcg_mini_sweep(cfg=QCG_CONFIG):
    data = DATASETS[cfg["subset"]]
    te = TEST_LOADERS[cfg["subset"]]
    rows = []
    for use_qcg in (False, True):
        for seed in cfg["seeds"]:
            mdl, meta = get_trained(cfg["model"], data, seed=seed,
                                    use_qcg=use_qcg,
                                    train_degrade_fn=mixed_degrade,
                                    tag="_mix", verbose=False)
            ss = meta["sigma_scale"]
            base = evaluate(mdl, te, sigma_scale=ss)["rmse"]
            for dim in cfg["dims"]:
                for sev in [0] + cfg["sevs"]:
                    gen = torch.Generator(device=DEVICE)
                    gen.manual_seed(500 + sev)
                    deg_fn = (lambda xb, g, d=dim, s=sev:
                              degrade(xb, d, s, g))
                    r = evaluate(mdl, te, deg_fn, gen, ss)
                    rows.append(dict(qcg=use_qcg, seed=seed, dim=dim,
                                     sev=sev, rmse=r["rmse"],
                                     rmse_base=base))
    return pd.DataFrame(rows)


QCG_SWEEP = qcg_mini_sweep()
QCG_SWEEP.to_csv(os.path.join(RUN_ROOT, "qcg_sweep_v3.csv"), index=False)


def qcg_elasticity_compare(df=QCG_SWEEP):
    """Per (dim): paired per-seed elasticity difference (QCG-on minus off)."""
    rows = []
    for dim in df["dim"].unique():
        diffs = []
        etas = {True: [], False: []}
        for seed in sorted(df["seed"].unique()):
            per = {}
            for flag in (False, True):
                g = df[(df.qcg == flag) & (df.seed == seed)
                       & (df.dim == dim)]
                agg = g.groupby("sev")["rmse"].mean().reset_index()
                if len(agg) >= 2:
                    per[flag] = np.polyfit(agg["sev"], agg["rmse"], 1)[0]
                    etas[flag].append(per[flag])
            if len(per) == 2:
                diffs.append(per[True] - per[False])
        diffs = np.array(diffs)
        p = (wilcoxon(diffs, alternative="less").pvalue
             if len(diffs) >= 3 and np.any(diffs != 0) else np.nan)
        rows.append(dict(dim=dim,
                         eta_qcg_off=round(float(np.mean(etas[False])), 2),
                         eta_qcg_on=round(float(np.mean(etas[True])), 2),
                         mean_diff=round(float(diffs.mean()), 2),
                         p_raw=float(p)))
    out = pd.DataFrame(rows)
    out["p_holm"] = holm(out["p_raw"].values)
    return out


QCG_RESULT = qcg_elasticity_compare()
QCG_RESULT.to_csv(os.path.join(RUN_ROOT, "qcg_result_v3.csv"), index=False)
print("QCG ablation — per-severity RMSE slope, gate on vs off "
      "(negative diff = gate helps):")
print(QCG_RESULT.round({"p_raw": 4, "p_holm": 4}).to_string(index=False))


## Section 8 — Publication Figures

300-DPI vector PDFs + PNG previews, Paul Tol colorblind-safe palette, Computer-Modern math, legends **outside** the axes, captions self-contained. Per-subset figure variants carry the subset suffix.

In [ ]:
# ============================================================================
# SECTION 8a — STYLE
# ============================================================================
import shutil

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

USE_TEX = shutil.which("latex") is not None
mpl.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 11, "font.family": "serif", "mathtext.fontset": "cm",
    "text.usetex": USE_TEX, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})
TOL = ["#4477AA", "#EE6677", "#228833", "#CCBB44", "#66CCEE", "#AA3377",
       "#BBBBBB", "#000000"]


def savefig(name):
    plt.savefig(os.path.join(FIGDIR, f"{name}.pdf"))
    plt.savefig(os.path.join(FIGDIR, f"{name}.png"))


print(f"Style set (usetex={USE_TEX}).")


In [ ]:
# ============================================================================
# SECTION 8b — CORE FIGURES (per subset) + CROSS-DATASET OVERLAY
# ============================================================================

def fig_dose_response(df, subset, name=None):
    """Dose-response, 95% CI bands, legend BELOW the axes (v3 polish)."""
    name = name or f"fig1_dose_response_{subset}"
    sub_df = df[df.subset == subset]
    dims = sorted(sub_df["dim"].unique())
    models = list(dict.fromkeys(sub_df["model"]))
    n_seeds = sub_df["seed"].nunique()
    fig, axes = plt.subplots(1, len(models),
                             figsize=(4.0 * len(models), 3.9), sharey=True)
    handles = []
    for ax, m in zip(np.atleast_1d(axes), models):
        s2 = sub_df[sub_df.model == m]
        for i, d in enumerate(dims):
            gg = (s2[s2.dim == d].groupby("one_minus_q")["rmse"]
                  .agg(["mean", "std"]).reset_index())
            ci = 1.96 * gg["std"] / np.sqrt(n_seeds)
            (line,) = ax.plot(gg["one_minus_q"], gg["mean"], "o-", ms=3.5,
                              lw=1.4, color=TOL[i % len(TOL)], label=d)
            ax.fill_between(gg["one_minus_q"], gg["mean"] - ci,
                            gg["mean"] + ci, color=TOL[i % len(TOL)],
                            alpha=0.15, lw=0)
            if ax is np.atleast_1d(axes)[0]:
                handles.append(line)
        ax.set_title(f"{m} ({subset})")
        ax.set_xlabel(r"Degradation $(1-q)$")
    np.atleast_1d(axes)[0].set_ylabel("Test RMSE (cycles)")
    fig.legend(handles, dims, loc="lower center", ncol=4, fontsize=7.5,
               frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout()
    savefig(name)
    plt.show()


def fig_elasticity(elast, subset, name=None):
    name = name or f"fig2_elasticity_{subset}"
    e2 = elast[elast.subset == subset]
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.0))
    for ax, col, ttl, fmt in [
            (axes[0], "elasticity",
             rf"Raw $\eta$ — {subset}", ".1f"),
            (axes[1], "elasticity_rel",
             rf"Normalized $\eta^{{rel}}$ — {subset}", ".2f")]:
        pivot = e2.pivot(index="model", columns="dim", values=col)
        sns.heatmap(pivot, annot=True, fmt=fmt, cmap="cividis", ax=ax,
                    cbar_kws={"shrink": 0.9})
        ax.set_title(ttl, fontsize=10)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=30)
        for lbl in ax.get_xticklabels():
            lbl.set_ha("right")
    fig.tight_layout()
    savefig(name)
    plt.show()


def fig_cross_dataset(df, dim="completeness_MNAR",
                      name="fig7_cross_dataset"):
    """C3 visual: one dimension's dose-response overlaid across subsets."""
    subsets = df["subset"].unique()
    models = list(dict.fromkeys(df["model"]))
    fig, axes = plt.subplots(1, len(models),
                             figsize=(4.0 * len(models), 3.6), sharey=False)
    for ax, m in zip(np.atleast_1d(axes), models):
        for k, sub in enumerate(subsets):
            g = df[(df.subset == sub) & (df.model == m) & (df.dim == dim)]
            gg = (g.groupby("one_minus_q")["rmse"]
                  .agg(["mean", "std"]).reset_index())
            base = g["rmse_base"].mean()
            ax.plot(gg["one_minus_q"], gg["mean"] / base, "o-", ms=3.5,
                    lw=1.5, color=TOL[k], label=sub)
        ax.set_title(m)
        ax.set_xlabel(r"Degradation $(1-q)$")
        ax.axhline(1.0, color="gray", lw=0.8, ls=":")
    np.atleast_1d(axes)[0].set_ylabel(
        r"RMSE / RMSE$_{clean}$")
    np.atleast_1d(axes)[-1].legend(frameon=False, fontsize=8)
    fig.suptitle(f"Cross-dataset dose-response — {dim}", y=1.02,
                 fontsize=11)
    fig.tight_layout()
    savefig(name)
    plt.show()


for _sub in CONFIG["subsets"]:
    fig_dose_response(SWEEP, _sub)
    fig_elasticity(ELAST, _sub)
if len(CONFIG["subsets"]) >= 2:
    fig_cross_dataset(SWEEP)


In [ ]:
# ============================================================================
# SECTION 8c — INTERACTIONS, CALIBRATION, BUDGET, CONVERGENCE,
#              INJECTION-POINT, QCG FIGURES
# ============================================================================

def fig_interactions(results, name="fig3_interactions"):
    fig, axes = plt.subplots(1, len(results),
                             figsize=(3.6 * len(results), 3.5), sharey=False)
    for ax, r in zip(np.atleast_1d(axes), results):
        df = r["per_seed"]
        means = df[["dA", "dB", "dAB"]].mean()
        stds = df[["dA", "dB", "dAB"]].std()
        vals = [means["dA"], means["dB"], means["dA"] + means["dB"],
                means["dAB"]]
        errs = [stds["dA"], stds["dB"],
                np.sqrt(stds["dA"] ** 2 + stds["dB"] ** 2), stds["dAB"]]
        ax.bar([r"$\Delta_A$", r"$\Delta_B$", r"$\Delta_A{+}\Delta_B$",
                r"$\Delta_{AB}$"], vals, yerr=errs, capsize=3,
               color=[TOL[0], TOL[0], TOL[2], TOL[1]])
        short_a = r["dim_a"].replace("completeness_", "").replace(
            "accuracy_", "")
        short_b = r["dim_b"].replace("completeness_", "").replace(
            "accuracy_", "")
        ax.set_title(f"{short_a} $\\times$ {short_b}\n{r['model']} | "
                     f"$p_{{Holm}}$={r['p_holm']:.3f}", fontsize=9)
        ax.set_ylabel(r"$\Delta$RMSE (cycles)")
    fig.tight_layout()
    savefig(name)
    plt.show()


def fig_calibration_prepost(subset="FD001", name_model="transformer",
                            name="fig4_calibration"):
    mdl = CLEAN[subset][name_model]["model"]
    scale = CLEAN[subset][name_model]["meta"]["sigma_scale"]
    plt.figure(figsize=(3.8, 3.8))
    plt.plot([0, 1], [0, 1], "k--", lw=1, label="ideal")
    for lab, ss, color in [("pre-calibration", 1.0, TOL[6]),
                           (f"post ($s$={scale:.2f})", scale, TOL[1])]:
        res = evaluate(mdl, TEST_LOADERS[subset], sigma_scale=ss)
        ps = np.linspace(0.05, 0.95, 10)
        emp = [np.mean(res["y"] <= res["mu"] + res["sigma"] * _norm.ppf(p))
               for p in ps]
        plt.plot(ps, emp, "o-", ms=4, color=color, label=lab)
    plt.xlabel("Nominal quantile")
    plt.ylabel("Empirical coverage")
    plt.legend(frameon=False, fontsize=8)
    plt.title(f"Calibration ({name_model}, {subset})")
    plt.tight_layout()
    savefig(name)
    plt.show()


def fig_iso_fidelity(subset="FD001", model_name="transformer", seed=0,
                     grid_sevs=(0, 1, 2, 3, 4, 5), floor_mult=1.3,
                     name="fig5_iso_fidelity"):
    mdl, meta = get_trained(model_name, DATASETS[subset], seed=seed,
                            verbose=False)
    ss = meta["sigma_scale"]
    te = TEST_LOADERS[subset]
    sevs = list(grid_sevs)
    z = np.zeros((len(sevs), len(sevs)))
    for i, s_comp in enumerate(sevs):
        for j, s_time in enumerate(sevs):
            def fn(xb, g, sa=s_comp, sb=s_time):
                u = torch.ones(xb.shape[0], xb.shape[2], device=xb.device)
                if sa:
                    xb, ua = degrade(xb, "completeness_MNAR", sa, g)
                    u = torch.minimum(u, ua)
                if sb:
                    xb, ub = degrade(xb, "timeliness", sb, g)
                    u = torch.minimum(u, ub)
                return xb, u
            gen = torch.Generator(device=DEVICE)
            gen.manual_seed(3)
            z[i, j] = evaluate(mdl, te, fn, gen, ss)["rmse"]
    plt.figure(figsize=(4.9, 3.9))
    cs = plt.contourf(sevs, sevs, z, levels=12, cmap="viridis")
    plt.colorbar(cs, label="RMSE (cycles)")
    plt.contour(sevs, sevs, z, levels=[z[0, 0] * floor_mult], colors="w",
                linewidths=2)
    plt.xlabel("Timeliness severity (ordinal 0–5)")
    plt.ylabel("Completeness (MNAR) severity (ordinal 0–5)")
    plt.title(f"Iso-fidelity budget surface — {model_name}, {subset}\n"
              f"white contour = {floor_mult:.1f}$\\times$ clean RMSE "
              f"({z[0, 0]:.1f})")
    plt.tight_layout()
    savefig(name)
    plt.show()


def fig_convergence(subset="FD001", name="fig6_convergence"):
    plt.figure(figsize=(5.2, 3.4))
    for i, m in enumerate(CONFIG["models"]):
        plt.plot(CLEAN[subset][m]["meta"]["history"], "-", lw=1.5,
                 color=TOL[i], label=m)
    plt.axvline(WARMUP_EPOCHS - 0.5, color="gray", ls=":", lw=1,
                label="MSE→NLL switch")
    plt.xlabel("Epoch")
    plt.ylabel("Validation RMSE (cycles)")
    plt.legend(frameon=False, fontsize=8)
    plt.title(f"Training convergence ({subset}, seed {GLOBAL_SEED})")
    plt.tight_layout()
    savefig(name)
    plt.show()


def fig_injection_points(ip=None, name="fig8_injection_points"):
    ip = IP if ip is None else ip
    agg = (ip.groupby(["dim", "sev"])[["d_inference", "d_train", "d_both"]]
           .agg(["mean", "std"]))
    cells = list(agg.index)
    width = 0.25
    xpos = np.arange(len(cells))
    plt.figure(figsize=(1.7 * len(cells) + 2, 3.6))
    for k, (col, lab) in enumerate([("d_inference", "inference"),
                                    ("d_train", "train"),
                                    ("d_both", "both")]):
        plt.bar(xpos + (k - 1) * width, agg[(col, "mean")], width,
                yerr=agg[(col, "std")], capsize=3, color=TOL[k], label=lab)
    labels = [f"{d.replace('completeness_', '').replace('accuracy_', '')}"
              f"\nsev {s}" for d, s in cells]
    plt.xticks(xpos, labels, fontsize=8)
    plt.ylabel(r"$\Delta$RMSE vs clean (cycles)")
    plt.legend(frameon=False, fontsize=8, title="injection point")
    plt.title("Lifecycle stage of the degradation matters")
    plt.tight_layout()
    savefig(name)
    plt.show()


def fig_qcg(result=None, name="fig9_qcg_ablation"):
    result = QCG_RESULT if result is None else result
    x = np.arange(len(result))
    width = 0.35
    plt.figure(figsize=(1.5 * len(result) + 2, 3.4))
    plt.bar(x - width / 2, result["eta_qcg_off"], width, color=TOL[6],
            label="gate off")
    plt.bar(x + width / 2, result["eta_qcg_on"], width, color=TOL[2],
            label="gate on")
    plt.xticks(x, [d.replace("completeness_", "").replace("accuracy_", "")
                   for d in result["dim"]], fontsize=8)
    plt.ylabel("RMSE slope per severity step")
    plt.legend(frameon=False, fontsize=8)
    plt.title("QCG ablation (matched mixed-degradation training)")
    plt.tight_layout()
    savefig(name)
    plt.show()


fig_interactions(INTERACTIONS)
fig_calibration_prepost()
fig_iso_fidelity()
fig_convergence()
fig_injection_points()
fig_qcg()


In [ ]:
# ============================================================================
# SECTION 8d — JOURNAL TABLES A-H (LaTeX / CSV / Word-ready HTML)
# ============================================================================

def _export(tbl, name, caption):
    tbl.to_csv(os.path.join(TBLDIR, f"table_{name}.csv"), index=False)
    tex_tbl = tbl.copy()
    for col in tex_tbl.columns:
        if tex_tbl[col].dtype == object:
            tex_tbl[col] = tex_tbl[col].map(
                lambda v: v.replace("_", r"\_")
                if isinstance(v, str) and "$" not in v else v)
    with open(os.path.join(TBLDIR, f"table_{name}.tex"), "w") as f:
        f.write(tex_tbl.to_latex(index=False, escape=False, caption=caption,
                                 label=f"tab:{name}"))
    tbl.to_html(os.path.join(TBLDIR, f"table_{name}.html"), index=False)


# A — baselines across subsets, pre/post calibration.
TBL_A = pd.DataFrame([
    dict(Subset=sub, Model=m.upper(), RMSE=round(v["rmse"], 2),
         NASA_Score=int(round(v["score"])),
         ECE_pre=round(v["ece_pre"], 3), ECE_post=round(v["ece_post"], 3),
         Cov90_pre=round(v["cov_pre"], 2), Cov90_post=round(v["cov_post"], 2))
    for sub in CONFIG["subsets"] for m, v in CLEAN[sub].items()])
_export(TBL_A, "A_baselines",
        "Clean-data fidelity across subsets with pre/post variance "
        "calibration (seed 42).")

# B — elasticity, raw + normalized, all subsets.
TBL_B = ELAST.copy()
TBL_B["CI95_raw"] = TBL_B.apply(
    lambda r: f"[{r.ci_lo:.2f}, {r.ci_hi:.2f}]", axis=1)
TBL_B["CI95_rel"] = TBL_B.apply(
    lambda r: f"[{r.rel_ci_lo:.2f}, {r.rel_ci_hi:.2f}]", axis=1)
TBL_B = (TBL_B[["subset", "model", "dim", "elasticity", "CI95_raw",
                "elasticity_rel", "CI95_rel"]]
         .rename(columns={"subset": "Subset", "model": "Model",
                          "dim": "DQ dimension",
                          "elasticity": "$\\eta$",
                          "elasticity_rel": "$\\eta^{rel}$"}))
TBL_B["$\\eta$"] = TBL_B["$\\eta$"].round(2)
TBL_B["$\\eta^{rel}$"] = TBL_B["$\\eta^{rel}$"].round(2)
_export(TBL_B, "B_elasticity",
        "DQ elasticity (raw and normalized) with bootstrap 95\\% CIs over "
        f"{len(CONFIG['seeds'])} seeds, per subset.")

# C — breakpoints + cross-dataset consistency.
_export(BREAKS, "C_breakpoints",
        "Segmented-regression breakpoints (NaN: adequately linear).")
if len(C3):
    _export(C3, "C3_cross_dataset",
            "Cross-dataset breakpoint consistency "
            f"(pre-set tolerance $\\pm{C3_TOL}$).")

# D — interactions with Holm correction and effect sizes.
TBL_D = INTER_DF.round({"mean_gap": 2, "p_raw": 4, "p_holm": 4,
                        "r_rb": 2})
_export(TBL_D, "D_interactions",
        "Interaction gaps with one-sided Wilcoxon $p$ (raw and "
        "Holm-adjusted) and matched-pairs rank-biserial effect size "
        f"(n={len(CONFIG['interaction_seeds'])} seeds).")

# E — signature confirmation (C4).
TBL_E = C4.round({"p_raw": 4, "p_holm": 4})
_export(TBL_E, "E_signatures",
        "Architecture elasticity-signature tests: discovery (FD001) vs. "
        "confirmation (FD004), paired Wilcoxon over seeds, Holm-adjusted.")

# F — injection-point study.
TBL_F = (IP.groupby(["dim", "sev"])[["d_inference", "d_train", "d_both"]]
         .agg(["mean", "std"]).round(2))
TBL_F.columns = ["_".join(c) for c in TBL_F.columns]
TBL_F = TBL_F.reset_index()
_export(TBL_F, "F_injection_points",
        "$\\Delta$RMSE by lifecycle injection point "
        "(train / inference / both), mean $\\pm$ std over seeds.")

# G — QCG ablation.
TBL_G = QCG_RESULT.round({"p_raw": 4, "p_holm": 4})
_export(TBL_G, "G_qcg",
        "Quality-Conditioned Gating ablation under matched "
        "mixed-degradation training (negative diff = gate reduces "
        "fragility).")

print("Exported tables A-G ->", TBLDIR)
print("\nTABLE A\n", TBL_A.to_string(index=False))
print("\nTABLE D\n", TBL_D.to_string(index=False))
print("\nTABLE E\n", TBL_E.to_string(index=False))


In [ ]:
# ============================================================================
# SECTION 8e — LIVE DQ MONITOR (pre-imputation observation)
# ============================================================================
TRAIN_STD = torch.from_numpy(
    DATASETS["FD001"]["Xtr"].std(axis=(0, 1))).to(DEVICE)


def estimate_quality(raw_window: torch.Tensor) -> dict:
    nan_frac = torch.isnan(raw_window).float().mean().item()
    q_completeness = 1.0 - nan_frac
    filled = _ffill(raw_window.unsqueeze(0)).squeeze(0)
    var_ratio = (filled.std(dim=0) / (TRAIN_STD + 1e-6)).mean().item()
    q_accuracy = float(np.clip(1.0 - abs(var_ratio - 1.0), 0.0, 1.0))
    obs = filled[~torch.isnan(raw_window).any(dim=1)]
    repeats = ((obs[1:] == obs[:-1]).float().mean().item()
               if len(obs) > 1 else 0.0)
    q_timeliness = float(np.clip(1.0 - repeats, 0.0, 1.0))
    return {"completeness": q_completeness, "accuracy": q_accuracy,
            "timeliness": q_timeliness}


def fidelity_confidence(q_hat, elast_df, subset="FD001",
                        model="transformer") -> float:
    sub = elast_df[(elast_df.model == model) & (elast_df.subset == subset)]
    eta = {"completeness": float(sub[sub.dim == "completeness_MNAR"]
                                 ["elasticity_rel"].mean()),
           "accuracy": float(sub[sub.dim == "accuracy"]
                             ["elasticity_rel"].mean()),
           "timeliness": float(sub[sub.dim == "timeliness"]
                               ["elasticity_rel"].mean())}
    total = sum(abs(v) for v in eta.values()) + 1e-9
    w = {k: abs(v) / total for k, v in eta.items()}
    return float(np.clip(1.0 - sum(w[k] * (1.0 - q_hat.get(k, 1.0))
                                   for k in w), 0.0, 1.0))


_win = torch.from_numpy(DATASETS["FD001"]["Xte"][0]).to(DEVICE)
_gen = torch.Generator(device=DEVICE)
_gen.manual_seed(11)
_raw_bad, _mask = make_missing(_win.unsqueeze(0), 4, "MNAR", _gen)
for label, w in [("clean", _win), ("degraded", _raw_bad.squeeze(0))]:
    q_hat = estimate_quality(w)
    conf = fidelity_confidence(q_hat, ELAST)
    q_str = ", ".join(f"{k}: {v:.2f}" for k, v in q_hat.items())
    print(f"{label:9s} q_hat={{ {q_str} }}  ->  "
          f"fidelity-confidence = {conf:.2f}")
print(f"(injected missing fraction: {_mask.float().mean().item():.2f})")


## Submission Checklist & Recommendations

**Target venues (in strategic order).** *ACM Journal of Data and Information Quality* (perfect scope fit for a DQ-taxonomy contribution), *Future Generation Computer Systems* (benchmark + systems angle), *Journal of Industrial Information Integration* (digital-twin community), *Data & Knowledge Engineering*. Cover-letter pitch: *"the first benchmark and metric framework that makes data quality a measurable, budgetable design parameter for digital twins."*

**Results-section skeleton (map figures/tables to claims):**
1. Framework & benchmark definition (taxonomy → operators → elasticity → budgets).
2. Clean baselines & training adequacy (Table A, Fig 6) — one honest sentence on compact-model vs SOTA accuracy.
3. Elasticity landscape (Fig 1–2, Table B): bias ≫ noise (C1), MNAR ≫ MCAR (C2), silent miscalibration (C6).
4. Collapse points & cross-dataset invariance (Table C/C3, Fig 7) — the "missingness cliff."
5. Interaction structure (Fig 3, Table D): mechanism-overlap explains sub- vs super-additivity (C5).
6. Architecture signatures, discovery→confirmation (Table E) — the honest successor to the failed uniform-PINN-robustness hypothesis; "no free lunch across DQ dimensions."
7. Lifecycle stage matters (Fig 8, Table F) and QCG ablation (Fig 9, Table G) — report the gate's result *whatever it is*.
8. DQ monitor demo + budget surface (Fig 5) as the practitioner payoff.

**Rigor items reviewers will check — all wired in:** multi-seed bootstrap CIs everywhere; Holm-corrected p-values within each test family; effect sizes (rank-biserial, Cliff's delta); pre-registered directions with both outcomes reportable; discovery/confirmation split across subsets; training-adequacy audit; per-epoch-resumable, seed-pinned, single-CONFIG pipeline. Release the Drive folder contents (minus checkpoints if size-constrained) + this notebook on GitHub/Zenodo and cite the DOI in the paper.

**Honesty lines to keep in the manuscript:** compact surrogates trade a few RMSE points vs. published SOTA for grid feasibility; elasticity is a local linear summary paired with breakpoint analysis wherever nonlinear; the severity→(1−q) mapping is operational and stated explicitly; C-MAPSS is simulated data — frame external validity accordingly and flag real-plant telemetry as future work.

**If a session dies:** `Runtime → Run all`. Everything resumes. To force-redo one unit of work, delete its checkpoint pair (or the relevant CSV rows); to redo everything, rename the `DQ4DT` Drive folder.